In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1: Extreme ESI Logistic Regressor with Decision Tree Imputer & Upsampling (`models/lr_extreme.ipynb`)

This notebook trains a **Layer 1 Multinomial Logistic Regressor** featuring **Decision Tree Imputation** and **Upsampling (`caret::upSample`)**:
- **Decision Tree Imputer Integration**: Loads the trained Decision Tree Regressor Imputer (`deploy/decision_tree_imputer.rds`) to predict and fill missing values in vital signs (`triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`).
- **Target Output**: Predicts whether a patient is **ESI 1** (Highest Acuity), **ESI 5** (Lowest Acuity), or **neither** (Intermediate ESI 2, 3, 4).
- **Class Imbalance Handling**: Applies `caret::upSample` to the training set so minority classes (ESI 1 & ESI 5) are balanced equally with the majority class ('neither').
- **Evaluation Metrics**: **Accuracy**, **Multi-Class ROC-AUC**, and **Log Loss**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(rpart)
library(nnet)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Load Decision Tree Imputer, & Fill Missing Vitals
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

base_features <- config$features$base_features
if (is.null(base_features)) {
  base_features <- config$features$data_name
}
target_col <- config$classes$target_col

df <- raw_df[, c(intersect(base_features, names(raw_df)), target_col)]

if ("gender" %in% names(df)) {
  df$gender <- ifelse(as.character(df$gender) == "Male", 1, 0)
}

# ---------------------------------------------------------
# Load Decision Tree Regressor Imputer
# ---------------------------------------------------------
imputer_path <- file.path("../deploy", "decision_tree_imputer.rds")
if (!file.exists(imputer_path)) imputer_path <- file.path("deploy", "decision_tree_imputer.rds")

impute_missing_vitals <- function(df, imputer_models) {
  df_imp <- df
  for (v_col in names(imputer_models)) {
    if (v_col %in% names(df_imp)) {
      na_idx <- which(is.na(df_imp[[v_col]]))
      if (length(na_idx) > 0) {
        preds <- predict(imputer_models[[v_col]], newdata = df_imp[na_idx, , drop = FALSE])
        df_imp[na_idx, v_col] <- preds
      }
    }
  }
  return(df_imp)
}

if (file.exists(imputer_path)) {
  imputer_obj <- readRDS(imputer_path)
  cat("Successfully loaded Decision Tree Imputer from:", imputer_path, "\n")
  df <- impute_missing_vitals(df, imputer_obj$models)
  cat("Missing vital values after decision tree imputation:", sum(is.na(df)), "\n")
} else {
  cat("Decision Tree Imputer not found at:", imputer_path, ". Omit NA rows as fallback...\n")
  df <- na.omit(df)
}

# ---------------------------------------------------------
# Create Layer 1 Target: '1', '5', or 'neither' (ESI 2, 3, 4)
# ---------------------------------------------------------
raw_esi <- as.character(df[[target_col]])
df$target_layer1 <- factor(ifelse(raw_esi == "1", "1",
                            ifelse(raw_esi == "5", "5", "neither")),
                           levels = c("1", "5", "neither"))

cat(sprintf("Layer 1 Dataset Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Layer 1 Target Distribution:\n")
print(table(df$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize numeric features (center and scale)
numeric_cols <- names(train_df)[sapply(train_df, is.numeric)]
preproc <- preProcess(train_df[, numeric_cols], method = c("center", "scale"))

train_df[, numeric_cols] <- predict(preproc, train_df[, numeric_cols])
val_df[, numeric_cols]   <- predict(preproc, val_df[, numeric_cols])
test_df[, numeric_cols]  <- predict(preproc, test_df[, numeric_cols])

cat(sprintf("Original Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Apply Upsampling & Train Layer 1 Logistic Regression Model
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names <- setdiff(names(train_df), c(target_col, "target_layer1"))

# Apply caret::upSample to balance minority classes ('1' and '5') in training data
cat("Applying caret::upSample to balance class distribution in training set...\n")
train_upsampled <- upSample(
  x = train_df[, feat_names, drop = FALSE],
  y = train_df$target_layer1,
  yname = "target_layer1"
)

cat(sprintf("Upsampled Training Set Ready: %d rows\n", nrow(train_upsampled)))
cat("Upsampled Training Target Distribution:\n")
print(table(train_upsampled$target_layer1))

formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))

cat("\nTraining Layer 1 Multinomial Logistic Regressor on Imputed & Upsampled Training Data...\n")
lr_model <- multinom(formula_lr, data = train_upsampled, trace = FALSE, MaxNWts = 5000)

cat("Layer 1 Logistic Regression training complete!\n")
print(summary(lr_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Scoring Metrics (Accuracy, ROC-AUC, Log Loss)
# ---------------------------------------------------------
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_layer1_lr <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data$target_layer1)
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   UPSAMPLED & IMPUTED LAYER 1 LOGISTIC REGRESSOR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss   : %.4f\n", log_loss))
  cat("\nConfusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Evaluate on Validation set
evaluate_layer1_lr(lr_model, val_df, "Validation")

# Evaluate on Test set
evaluate_layer1_lr(lr_model, test_df, "Test")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Layer 1 Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_extreme_model.rds")
saveRDS(list(model = lr_model, preproc = preproc), file = model_path)
cat("Layer 1 Extreme Logistic Regressor model saved to:", model_path, "\n")